# 07 - Attention and Transformers

**Repo:** ai-learning-notes | **Folder:** llm-concepts

## What this notebook covers

- Why attention was invented - the problem with RNNs
- What self-attention actually computes - Q, K, V explained without matrix algebra
- Multi-head attention - why one attention pass is not enough
- The transformer architecture - encoder, decoder, encoder-decoder
- Positional encoding - how transformers know word order
- How GPT-style (decoder-only) models generate text
- Why any of this matters for practical LLM use

**Pure Python only - no external dependencies.**

---
## 1. The Problem That Attention Solved

Before transformers, sequence models used RNNs (Recurrent Neural Networks).
An RNN reads text one token at a time, left to right, maintaining a hidden state.

**The vanishing gradient problem:**
To understand the word 'it' in:
'The contract was signed by the vendor. The vendor then invoiced the client. It was processed.'

The RNN has to carry information about 'contract' through 10+ steps of hidden state.
By the time it reaches 'it', the gradient signal from 'contract' has faded almost to zero.
The model effectively forgets what happened early in the sequence.

**The attention insight (Vaswani et al. 2017 - 'Attention Is All You Need'):**
Instead of reading sequentially, let every token look directly at every other token simultaneously.
No distance penalty. 'it' can attend directly to 'contract' regardless of how many tokens are between them.

This one idea obsoleted RNNs for most NLP tasks within two years.

In [ ]:
# Illustrating why RNNs fail on long sequences

def rnn_information_retention(sequence, decay_per_step=0.85):
    '''
    Simulates how much of token[0] information survives to each position.
    Real RNNs use learned gates (LSTM/GRU) but the fundamental decay problem persists.
    '''
    retention = 1.0
    retentions = [retention]
    for _ in range(len(sequence) - 1):
        retention *= decay_per_step
        retentions.append(retention)
    return retentions

sequence = [
    "The",     # 0 - we want to track this word
    "contract", # 1 - KEY WORD
    "was",      # 2
    "signed",   # 3
    "by",       # 4
    "the",      # 5
    "vendor",   # 6
    "who",      # 7
    "then",     # 8
    "invoiced", # 9
    "the",      # 10
    "client",   # 11
    "and",      # 12
    "it",       # 13 - needs to know about "contract"
]

retentions = rnn_information_retention(sequence[1:], decay_per_step=0.82)

print("RNN: how much 'contract' information survives to each later token")
print("=" * 55)
print(f"{"Position":<12} {"Token":<12} {"Info retained":>14}")
print("-" * 40)
for i, (token, ret) in enumerate(zip(sequence[1:], retentions)):
    bar = int(ret * 20) * chr(9608)
    flag = " <- target" if token == "it" else ""
    print(f"{i+1:<12} {token:<12} {ret:>8.1%}  {bar}{flag})

print()
print("Attention: 'it' attends directly to 'contract' with no decay.")
print("Distance between tokens is irrelevant.")

---
## 2. Self-Attention - What It Actually Computes

Self-attention answers one question for each token:
**Which other tokens in this sequence are most relevant to understanding me?**

Three learned projections per token:
- **Q (Query):** What am I looking for?
- **K (Key):** What do I contain / advertise?
- **V (Value):** What information do I actually provide if attended to?

The mechanism:
1. Compute dot product of Q_i with every K_j - this gives a raw attention score
2. Scale by sqrt(dimension) to prevent vanishing gradients
3. Softmax across all scores - converts to attention weights that sum to 1
4. Weighted sum of all V_j using those weights - this is the output for token i

Informally: each token broadcasts a query ('I am looking for the subject of this action'),
every other token responds with its key ('I am a noun, specifically a legal document'),
and the highest-matching key wins the most attention weight,
causing that token's value to dominate the output.

In [ ]:
import math

def dot_product(a, b):
    return sum(x * y for x, y in zip(a, b))

def softmax(scores):
    max_s = max(scores)
    exps = [math.exp(s - max_s) for s in scores]
    total = sum(exps)
    return [e / total for e in exps]

def self_attention(queries, keys, values, dim):
    '''
    Simplified self-attention for a sequence of tokens.
    queries, keys, values: list of vectors (one per token)
    Returns: list of output vectors (one per token)
    '''
    scale = math.sqrt(dim)
    outputs = []
    all_weights = []
    for q in queries:
        scores = [dot_product(q, k) / scale for k in keys]
        weights = softmax(scores)
        all_weights.append(weights)
        out = [sum(w * v[d] for w, v in zip(weights, values)) for d in range(dim)]
        outputs.append(out)
    return outputs, all_weights

# Simulated 4D Q/K/V vectors for a 5-token sequence
# In reality: 768D vectors learned during pretraining
tokens = ['The', 'contract', 'was', 'signed', 'it']

# Simulate: 'it' has a query that matches 'contract' key strongly
queries = [
    [0.1, 0.2, 0.3, 0.1],  # The
    [0.8, 0.7, 0.1, 0.2],  # contract
    [0.1, 0.1, 0.9, 0.1],  # was
    [0.1, 0.2, 0.8, 0.1],  # signed
    [0.8, 0.7, 0.1, 0.2],  # it - query similar to 'contract'
]
keys = [
    [0.1, 0.1, 0.2, 0.9],  # The
    [0.9, 0.8, 0.1, 0.1],  # contract - key matches 'it' query
    [0.1, 0.1, 0.8, 0.2],  # was
    [0.1, 0.2, 0.7, 0.3],  # signed
    [0.3, 0.4, 0.1, 0.8],  # it
]
values = [
    [0.1, 0.0, 0.1, 0.0],
    [0.9, 0.8, 0.1, 0.0],  # contract value - high information
    [0.0, 0.1, 0.8, 0.0],
    [0.0, 0.1, 0.7, 0.2],
    [0.3, 0.3, 0.1, 0.9],
]

outputs, weights = self_attention(queries, keys, values, dim=4)

print("SELF-ATTENTION WEIGHTS")
print("How much each token attends to every other token")
print("=" * 60)
print(f"{"":<10}", end="")
for t in tokens:
    print(f"{t:>10}", end="")
print()
print("-" * 60)
for i, (token, row) in enumerate(zip(tokens, weights)):
    print(f"{token:<10}", end="")
    for w in row:
        bar = ">" * int(w * 8)
        print(f"{w:>7.3f} {bar:<3}", end="")
    print()
print()
print("Key observation: 'it' attends most strongly to 'contract'")
print("because their Q and K vectors have the highest dot product.")
print("This is how transformers resolve pronoun reference across distance.")

---
## 3. Multi-Head Attention

A single attention pass computes one type of relationship between tokens.
But natural language has many simultaneous relationship types:
- Syntactic: which noun is this verb's subject?
- Semantic: which words share meaning?
- Coreference: which pronoun refers to which entity?
- Position: which token immediately follows this one?

Multi-head attention runs H independent attention passes in parallel,
each with different learned Q/K/V projection matrices.
Each head specialises in a different relationship type through training.
The H outputs are concatenated and projected back to the original dimension.

GPT-3: 96 attention heads per layer, 96 layers.
Every forward pass runs 96 × 96 = 9,216 attention computations.
Each head has 12,288 / 96 = 128 dimensions.

In [ ]:
# Multi-head attention - concept illustration

class AttentionHead:
    def __init__(self, head_id, specialisation):
        self.head_id = head_id
        self.specialisation = specialisation

    def attend(self, token_pair):
        '''Simulate what each head specialises in.'''
        token_a, token_b = token_pair
        # Different heads notice different things
        if 'syntactic' in self.specialisation:
            return 0.9 if (token_a == 'signed' and token_b == 'contract') else 0.1
        elif 'coreference' in self.specialisation:
            return 0.9 if (token_a == 'it' and token_b == 'contract') else 0.1
        elif 'position' in self.specialisation:
            return 0.8 if token_b == 'was' else 0.1
        return 0.3

heads = [
    AttentionHead(0, "syntactic subject-verb"),
    AttentionHead(1, "coreference resolution"),
    AttentionHead(2, "position adjacency"),
    AttentionHead(3, "semantic similarity"),
]

token_pairs = [
    ("signed", "contract"),   # verb -> subject
    ("it",     "contract"),   # pronoun -> antecedent
    ("was",    "signed"),     # position check
    ("it",     "signed"),     # alternative resolution
]

print("MULTI-HEAD ATTENTION - what each head notices")
print("=" * 65)
print(f"{"Token pair":<22}", end="")
for h in heads:
    print(f"Head {h.head_id}  ", end="")
print()
print(f"{"":<22}", end="")
for h in heads:
    print(f"{h.specialisation[:8]:<10}", end="")
print()
print("-" * 65)
for pair in token_pairs:
    print(f"{' -> '.join(pair):<22}", end="")
    for h in heads:
        score = h.attend(pair)
        bar = chr(9608) * int(score * 6)
        print(f"{score:.2f} {bar:<6}", end="")
    print()

print()
print("Head 0 specialises in grammar. Head 1 in pronouns. No head sees everything.")
print("Together they capture all relationship types simultaneously.")
print("This is why removing attention heads degrades specific NLP capabilities.")

---
## 4. The Transformer Architecture

A transformer layer stacks two sub-layers:

```
Input tokens
     |
 [Multi-Head Self-Attention]  <- tokens attend to each other
     |  (+ residual connection)
 [Layer Normalisation]
     |
 [Feed-Forward Network]       <- each token processed independently
     |  (+ residual connection)
 [Layer Normalisation]
     |
Output tokens (same shape as input)
```

This single layer is stacked N times (GPT-2: 12 layers, GPT-3: 96 layers, GPT-4: estimated 120+).

**Residual connections** (the + skip connections) are critical.
They let gradients flow directly through the network during training,
making it possible to train very deep models without vanishing gradients.
Without residuals, a 96-layer transformer would not train at all.

**Three architecture variants:**

| Type | Attention direction | Used for | Examples |
|---|---|---|---|
| Encoder-only | Bidirectional (sees all tokens) | Classification, embedding | BERT, nomic-embed-text |
| Decoder-only | Causal (sees only past tokens) | Text generation | GPT, Claude, Llama |
| Encoder-decoder | Enc: bidirectional, Dec: causal | Translation, summarisation | T5, BART |

In [ ]:
# The three architecture variants and their attention masks

def visualise_attention_mask(mask_type, n_tokens=5):
    '''
    Shows which positions can attend to which.
    1 = can attend, 0 = masked out
    '''
    mask = []
    for i in range(n_tokens):
        row = []
        for j in range(n_tokens):
            if mask_type == 'bidirectional':    # encoder-only
                row.append(1)                   # every token sees every other
            elif mask_type == 'causal':         # decoder-only
                row.append(1 if j <= i else 0)  # only sees past + self
            else:
                row.append(1)
        mask.append(row)
    return mask

tokens = ['The', 'contract', 'was', 'signed', 'it']

for mask_type, name, example in [
    ("bidirectional", "Encoder-only (BERT, nomic-embed-text)", "Sees ALL tokens - good for understanding"),
    ("causal",        "Decoder-only (GPT, Claude, Llama)",     "Sees only PAST tokens - required for generation"),
]:
    mask = visualise_attention_mask(mask_type, len(tokens))
    print(f"\n{name}")
    print(f"  {example}")
    print()
    header = "          " + "".join(f"{t[:6]:>8}" for t in tokens)
    print(header)
    print("  " + "-" * (len(header) - 2))
    for i, (token, row) in enumerate(zip(tokens, mask)):
        cells = "".join(chr(9608)*6 if v else "  .   " for v in row)
        print(f"  {token:<8}  {cells}")

print()
print("Claude is decoder-only: when generating token N, it only sees tokens 0..N-1.")
print("nomic-embed-text is encoder-only: sees the full sequence for rich embeddings.")
print("This is why you cannot use a generation model as a drop-in embedding model.")

---
## 5. Positional Encoding

Self-attention is permutation-invariant.
If you shuffle the tokens, the attention weights change but the mechanism has no built-in
sense of position. 'The dog bit the man' and 'The man bit the dog' would be indistinguishable.

Transformers solve this by adding a positional encoding to each token embedding before attention.

**Two approaches:**

**Absolute (original Transformer, GPT-2):**
Add a fixed sinusoidal signal to each position. Position 0 gets one pattern, position 1 another.
Simple but cannot generalise to sequences longer than training length.

**Rotary (RoPE - GPT-NeoX, Llama, Mistral):**
Rotate Q and K vectors by an angle proportional to position before computing dot products.
The dot product Q·K then naturally encodes relative position.
Generalises better to longer sequences. Used in most modern open-source LLMs.

**Why it matters for you:**
Context window limits exist because positional encoding degrades beyond the training length.
A model trained on 4K tokens sees position 5000 as an extrapolation it was never trained to handle.
This is why Claude performs worse on very long documents even within the 200K context window.

In [ ]:
import math

def sinusoidal_encoding(position, dim=8):
    '''
    Original transformer positional encoding (Vaswani et al. 2017).
    Each dimension oscillates at a different frequency.
    '''
    encoding = []
    for i in range(dim):
        if i % 2 == 0:
            encoding.append(math.sin(position / (10000 ** (i / dim))))
        else:
            encoding.append(math.cos(position / (10000 ** ((i-1) / dim))))
    return encoding

print('SINUSOIDAL POSITIONAL ENCODING')
print('Each position gets a unique pattern across dimensions')
print('=' * 65)
print(f"{'Position':<10}", end='')
for i in range(8):
    print(f"  dim{i}", end="")
print()
print("-" * 65)
for pos in [0, 1, 2, 5, 10, 50, 100]:
    enc = sinusoidal_encoding(pos)
    formatted = ''.join(f'{v:+.2f}' for v in enc)
    print(f"{pos:<10} {formatted}")

print()
print('POSITIONAL SIMILARITY - do nearby positions have similar encodings?')
print("-" * 50)
def cosine_sim(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    na = math.sqrt(sum(x**2 for x in a))
    nb = math.sqrt(sum(x**2 for x in b))
    return dot/(na*nb) if na and nb else 0

base = sinusoidal_encoding(0)
for pos in [1, 5, 10, 50, 100, 500]:
    sim = cosine_sim(base, sinusoidal_encoding(pos))
    bar = chr(9608) * int(sim * 20)
    print(f"  pos 0 vs pos {pos:<5}: {sim:.4f}  {bar}")

print()
print("Nearby positions are more similar. Distant positions diverge.")
print("RoPE (used in Llama, Mistral) applies this as a rotation on Q/K vectors")
print("instead of additive embeddings - better generalisation to long sequences.")

---
## 6. How GPT-Style Models Generate Text

GPT models (and Claude) are decoder-only transformers.
Generation works by repeating one simple step:

```
Prompt tokens -> Transformer layers -> Logits over vocabulary (50,000+ tokens)
                                              |
                                    Sample one token
                                              |
                              Append to sequence, repeat
```

**What 'logits' are:**
The final transformer layer outputs a vector of raw scores, one per vocabulary token.
These are called logits. Higher = more likely next token.
Softmax converts them to a probability distribution.

**Temperature controls the distribution shape:**
- T=0: always pick the highest logit (greedy, deterministic)
- T=1: sample from the raw softmax distribution
- T>1: flatten the distribution (more random, more creative)
- T<1: sharpen the distribution (more focused, less varied)

This connects directly to notebook 02 (temperature and sampling) —
now you know what temperature is actually modifying.

In [ ]:
import math, random

def apply_temperature(logits, temperature):
    if temperature == 0:
        # Greedy: return one-hot on argmax
        max_idx = logits.index(max(logits))
        return [1.0 if i == max_idx else 0.0 for i in range(len(logits))]
    scaled = [l / temperature for l in logits]
    max_s = max(scaled)
    exps = [math.exp(s - max_s) for s in scaled]
    total = sum(exps)
    return [e / total for e in exps]

# Simulated logits for next token after 'The contract was'
vocab = ['signed', 'approved', 'violated', 'negotiated', 'renewed', 'amended', 'the', 'a']
logits = [4.2, 3.8, 2.1, 3.0, 2.5, 2.8, 1.2, 0.8]

print("TEMPERATURE EFFECT ON TOKEN PROBABILITY DISTRIBUTION")
print("Prompt: 'The contract was ...'")
print("=" * 65)

for temp in [0.0, 0.5, 1.0, 1.5, 2.0]:
    probs = apply_temperature(logits, temp)
    label = f"T={temp}"
    if temp == 0.0: label += " (greedy)"
    if temp == 1.0: label += " (default)"
    print(f"\n{label}:")
    for token, prob in zip(vocab, probs):
        bar = chr(9608) * int(prob * 40)
        print(f"  {token:<14} {prob:.4f}  {bar}")

print()
print("T=0:   always picks 'signed' - deterministic, repetitive")
print("T=1:   'signed' likely but 'approved' and others possible")
print("T=2:   distribution flattens - 'violated', 'amended' become plausible")
print()
print("For enterprise RAG (local-rag-llm): use T=0 or T=0.1")
print("For creative writing: T=0.8-1.2")
print("For brainstorming: T=1.2-1.5")

---
## 7. Why This Matters for Practical LLM Use

You do not need to implement transformers. But understanding the internals
changes how you design prompts, RAG systems, and agents.

| Mechanism | Practical implication |
|---|---|
| Attention is O(n²) in sequence length | Long contexts are expensive - costs scale quadratically |
| Decoder-only sees only past tokens | You cannot ask Claude to 'go back and fix line 3' mid-generation |
| Residual connections preserve early info | System prompt is not forgotten - it flows through every layer |
| Positional encoding degrades at length | Performance on very long docs degrades even within context limit |
| Temperature modifies logit distribution | T=0 is deterministic and best for factual/structured tasks |
| Multi-head attention has fixed heads | Different heads capture syntax, semantics, coreference separately |
| Embedding models are encoder-only | nomic-embed-text sees the full chunk - this is why mean pooling works |

**The most important insight for RAG:**
Claude's attention can span your full retrieved context, but:
- Information at the middle of a long context gets less attention than start/end
- This is the 'lost in the middle' problem (notebook 06 in this series)
- Short, focused chunks (your CrossEncoder reranking to top 5) directly combat this
- Your local-rag-llm architecture is correct for this reason

In [ ]:
# Connecting transformer internals to local-rag-llm architecture decisions

architecture_decisions = [
    {
        "decision": "chunk_size=2000, top_k=5",
        "transformer_reason": "Lost in the middle: attention degrades on long middle content.",
        "effect": "5 focused chunks beat 20 chunks where 15 are ignored.",
    },
    {
        "decision": "CrossEncoder reranking before sending to Claude",
        "transformer_reason": "Encoder-only CrossEncoder reads Q+chunk jointly via bidirectional attention.",
        "effect": "More precise relevance scoring than bi-encoder which encodes separately.",
    },
    {
        "decision": "nomic-embed-text for embeddings (encoder-only)",
        "transformer_reason": "Encoder-only sees full chunk via bidirectional attention -> richer embedding.",
        "effect": "Embedding captures full semantic context, not just left-to-right.",
    },
    {
        "decision": "Claude API temperature=0 for RAG answers",
        "transformer_reason": "T=0 = greedy decoding, always picks highest logit.",
        "effect": "Deterministic factual answers, no creative hallucination.",
    },
    {
        "decision": "system prompt enforces context-only answering",
        "transformer_reason": "Residual connections mean system prompt flows through all 96 layers.",
        "effect": "Constraint is active throughout generation, not just at start.",
    },
]

print("HOW TRANSFORMER INTERNALS JUSTIFY local-rag-llm DESIGN CHOICES")
print("=" * 70)
for d in architecture_decisions:
    print(f"\nDecision:  {d['decision']}")
    print(f"Why:       {d['transformer_reason']}")
    print(f"Effect:    {d['effect']}")

---
## 8. Summary

| Concept | Key point |
|---|---|
| **RNN problem** | Information decays over distance - attention fixes this |
| **Q, K, V** | Query = what I look for, Key = what I advertise, Value = what I provide |
| **Self-attention** | Every token attends to every other - no distance penalty |
| **Attention score** | dot(Q, K) / sqrt(dim) -> softmax -> weighted sum of V |
| **Multi-head** | H parallel attention passes, each specialises in a different relationship |
| **Encoder-only** | Bidirectional attention, used for embeddings (BERT, nomic-embed-text) |
| **Decoder-only** | Causal attention, used for generation (GPT, Claude, Llama) |
| **Residual connections** | Skip connections that let gradients flow through deep networks |
| **Positional encoding** | Sinusoidal (GPT-2) or RoPE (Llama) - injects token position into embeddings |
| **Temperature** | Scales logits before softmax - T=0 greedy, T>1 random |
| **Lost in the middle** | Attention degrades on long middle context - use short focused chunks |

---
## Next Notebooks

- **08** - Hallucination - causes, types, and how to reduce it
- **09** - Structured outputs and JSON mode
- **10** - Tool use and function calling
- **11** - LLM evaluation metrics